# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第1章：環境構築とデータの読み込み：RとGoogle Colabで最初の一歩


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


本文（KDP側）第1章に対応。tidyverseの基本操作とVOTデータの可視化を行う。

■ パッケージのインストールと読み込み：

初回のみ数分かかる。`tidyverse` は `readr`（CSV読込）/`dplyr`（データ整形）/`tidyr`（縦横変換）/`ggplot2`（作図）をまとめたメタパッケージ。

In [ ]:
# install.packages() はパッケージを一度だけダウンロード・インストールする。
# 毎回のセッションで必要なのは library() の方だが、Colabは実行環境が
# セッションごとにリセットされるため、このノートブックでは毎回両方書いてある。
install.packages("tidyverse", repos = "https://cloud.r-project.org")
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


■ 擬似VOTデータの生成：

Lisker & Abramson (1964) の古典的な多言語VOT研究は、破裂音の有声/無声の対立が、言語によって異なる位置で区切られる3つの音響カテゴリ（有声先行、無声無気、無声有気）に整理できることを示した。以来、VOT（Voice Onset Time、破裂音の開放から声帯振動開始までの時間）は音声学で最も頻繁に測定される指標の一つになっている。

本書では実録音の代わりに、Baayenら (2008) やWinter (2019) が例示する典型的な有声/無声破裂音のVOT分布（有声: 平均15ms前後、無声: 平均70-80ms前後）を模した擬似データを使う。理由は次の2点。
1. 著作権・被験者同意の問題なしに、読者全員が同じデータで再現できる
2. 効果をあえて大きめに設定してあるので、モデルの出力が読み取りやすい

実データに差し替えたい場合は、下のコードの `vot_data <- tibble(...)` を自分の `read_csv("手元のファイル.csv")` に置き換えるだけでよい。

In [ ]:
set.seed(42)  # 乱数のseedを固定する。同じ数字なら誰が実行しても同じ結果になる
n_speaker <- 10                    # 話者数
n_word    <- 20                    # 単語数（各話者が全単語を1回ずつ発話する）
n_obs     <- n_speaker * n_word    # = 200行

# 何行目がどの話者・どの単語かを表す添字。あとで話者ごと・単語ごとの
# 値を引くのに使う（speaker_intercept[speaker_idx] のように書ける）。
speaker_idx <- rep(1:n_speaker, each  = n_word)
word_idx    <- rep(1:n_word,    times = n_speaker)

# 1行ごとにvoiced/voicelessを交互に並べる。話者1人あたり20行なので、
# どの話者を見ても必ずvoiced10回・voiceless10回が含まれる
# ——これは第5章で「話者ごとにconditionの効果を推定する」ために必須の設計。
condition_vec <- rep(c("voiced", "voiceless"), times = n_obs / 2)

# ---- 話者差・単語差を「先に」作っておく ----
# 実際の音声データでは、同じ条件でも話者によってVOTの平均が違い（声道長・
# 発話速度・方言）、単語によっても違う（後続母音・語長・頻度）。
# 擬似データでもこの構造を明示的に作っておかないと、第5章以降で使う
# 混合効果モデルが「推定すべきばらつきが最初から存在しない」データを
# 相手にすることになり、分散推定値がすべて0（singular fit）になってしまう。
speaker_intercept <- rnorm(n_speaker, mean = 0, sd = 6)
# 話者ごとのVOTの底上げ量。どの条件にも一律にかかる（＝ランダム切片）。
speaker_slope <- rnorm(n_speaker, mean = 0, sd = 10)
# 話者ごとの「有声と無声をどれだけ大きく作り分けるか」の差（＝ランダム傾き）。
# 話者間のばらつきは無声（VOTが長い側）で特に大きいことが知られているので、
# voiceless条件にだけ加える。
word_intercept <- rnorm(n_word, mean = 0, sd = 3)
# 単語ごとの癖（＝項目のランダム切片）。

vot_data <- tibble(
  speaker   = paste0("S", sprintf("%02d", speaker_idx)),
  word      = paste0("word_", word_idx),
  gender    = rep(rep(c("M", "F"), each = n_speaker / 2), each = n_word),
  condition = condition_vec,
  # VOTは「固定効果 + 話者差 + 単語差 + 残差」を足し合わせて作る。
  # これは第5章で当てはめる混合効果モデルの式そのものであり、
  # モデルが推定した分散をこの生成時の値と見比べられるようにしてある。
  VOT = 20 +                                              # 有声条件の基準値
    if_else(condition == "voiceless", 55, 0) +            # 無声条件の上乗せ（→約75ms）
    speaker_intercept[speaker_idx] +                      # 話者のランダム切片
    if_else(condition == "voiceless",                     # 話者のランダム傾き
            speaker_slope[speaker_idx], 0) +
    word_intercept[word_idx] +                            # 単語のランダム切片
    # 残差。無声条件のほうがばらつきが大きい（実際のVOTデータもそうなっている）。
    # 第3章のモデル診断で「等分散性が崩れている」例として使う。
    rnorm(n_obs, mean = 0, sd = if_else(condition == "voiced", 4, 8)),
  F0 = rnorm(n_obs, mean = 130, sd = 15)  # F0はここでは条件と無関係に生成
)

# ★注意: VOTを c(rnorm(100, 15, 4), rnorm(100, 75, 10)) のように
# 「前半100行・後半100行」で単純に連結してはいけない。上のcondition_vecは
# 1行ごとに交互なので、連結方式だと条件とVOTがズレ、実際にはgender
# （前半100行=M、後半100行=F）とVOTの高低が一致してしまうというバグになる
# ——実際にこれで一度失敗した。必ず if_else(condition == ..., ...) の形で
# 「condition列を見てから値を選ぶ」実装にすること。

head(vot_data)  # 先頭6行を確認。long format（1行1観測）になっているか目視でチェック
write_csv(vot_data, "vot_f0_synthetic.csv")  # Colabの実行環境（一時領域）にファイルとして保存。
# 左のフォルダアイコンから「vot_f0_synthetic.csv」がダウンロードできる。
# 第2章以降のノートブックはこのファイルを読み込むのではなく、同じset.seed(42)で
# その場で作り直す設計にしてある（理由は各章の「共通データセットの読み込み」の
# コメントを参照）。ここで保存するのは、あくまで自分の手元に残す・別のツールで
# 開いて確認するためのものである。

# --- 実行結果の例（先頭6行。実際にRで実行して確認済み）---
# # A tibble: 6 x 6
#   speaker word   gender condition   VOT    F0
#   <chr>   <chr>  <chr>  <chr>     <dbl> <dbl>
# 1 S01     word_1 M      voiced     28.1  119.
# 2 S01     word_2 M      voiceless  88.0  145.
# 3 S01     word_3 M      voiced     30.7  149.
# 4 S01     word_4 M      voiceless  94.1  149.
# 5 S01     word_5 M      voiced     28.4  109.
# 6 S01     word_6 M      voiceless  98.4  161.
# データ全体では voiced が約6〜38ms、voiceless が約33〜109msの範囲に散らばる
# （話者差・単語差を足しているぶん、条件の平均値20/75msの前後に幅が出る）。
# ここでvoiced行のVOTがvoiceless行よりはっきり小さいことを確認してから先に進むこと。
# もしvoiced行に90msのような値が混ざっていたら、condition列とVOT列がズレている証拠。
#
# なお、S01の6行はどれも条件の平均より高め（voiced 28ms前後）に出ているが、
# これは不具合ではなく speaker_intercept[1] が正の値だったためである。
# 話者ごとに水準が違うことこそ、第5章で混合効果モデルを使う理由そのものになる。


■ 記述統計の集計（dplyr）：

In [ ]:
summary_stats <- vot_data %>%             # パイプ演算子 %>% は「左の結果を右の関数の第一引数に渡す」
  group_by(condition, gender) %>%          # condition×genderの組み合わせごとにグループ化
  summarise(
    mean_VOT = mean(VOT),
    sd_VOT = sd(VOT),
    median_VOT = median(VOT),
    n = n(),                                # そのグループの観測数
    .groups = "drop"                        # 集計後にグループ化情報を残さない（後続処理の事故防止）
  )

print(summary_stats)

# --- 実行結果 ---
# # A tibble: 4 x 6
#   condition gender mean_VOT sd_VOT median_VOT     n
#   <chr>     <chr>     <dbl>  <dbl>      <dbl> <int>
# 1 voiced    F          23.6   6.25       22.9    50
# 2 voiced    M          22.1   6.43       22.1    50
# 3 voiceless F          70.1  17.5        73.4    50
# 4 voiceless M          80.6  15.6        79.0    50
#
# 【出力の読み方】
# ・1行が「condition×genderの1組み合わせ」を表す。2条件×2性別で4行になる。
#   n列が全て50なのは、10話者×20語=200行が4群に均等に割れている証拠である。
#   ここが不揃いなら、集計より先にデータの欠けを疑う。
# ・列の下の <chr> <dbl> <int> は型の表示。<chr>は文字列、<dbl>は小数を持つ数値、
#   <int>は整数。**数値であるべき列が<chr>になっていたら、そこが読み込みミス**で、
#   平均も計算できない。表を見るときは値より先に型を見る癖をつける。
# ・mean_VOTを見ると、voicedが約22〜24ms、voicelessが約70〜81msで、
#   条件による差（約50ms）が一目でわかる。これが本書で一貫して追う効果である。
# ・sd_VOTはvoicedで約6ms、voicelessで約16〜18msと、**無声条件のほうが3倍近く
#   ばらついている**。これは実際のVOTデータにも見られる性質で、第3章の
#   モデル診断で「等分散性が崩れている」例として再登場する。
# ・mean_VOTとmedian_VOTがどの行でも近い（例: 70.1と73.4）。大きく食い違って
#   いれば分布が歪んでいる合図だが、ここでは概ね対称とみてよい。
#
# ★ここで早合点しやすい点: voiceless行のFが70.1、Mが80.6と10msも違うため、
#  「性別でVOTが違う」と読みたくなる。しかし第1章のデータ生成コードに
#  genderの効果は一切入れていない。S01〜S05をM、S06〜S10をFに割り当てて
#  いるので、これは**たまたまM側に「有声と無声を大きく作り分ける話者」が
#  多く入った**というだけである。10人を5人ずつに分けた比較でしかないことに
#  注意する。第5章で話者のランダム効果を入れると、この見かけの差は
#  話者間のばらつきとして吸収される。


■ データハンドリングの基本（dplyrの主要な動詞）：

`group_by()` と `summarise()` はすでに使ったが、実際のデータ整形では他にもいくつかの基本操作を組み合わせることになる。ここでまとめて練習する。**どれも「元のvot_dataを直接書き換えない」**（`<-` で新しい名前に代入するか、パイプの途中で使い捨てる）。元データを保っておけば、途中で間違えても最初からやり直せる。

In [ ]:
# filter(): 行を条件で選ぶ。「無声条件で、かつVOTが100ms超」の行だけを見る。
vot_data %>%
  filter(condition == "voiceless", VOT > 100) %>%
  select(speaker, word, VOT)   # select(): 見たい列だけに絞る

# --- 実行結果 ---
# # A tibble: 3 x 3
#   speaker word     VOT
#   <fct>   <fct>  <dbl>
# 1 S01     word_8  103.
# 2 S02     word_4  109.
# 3 S02     word_6  104.
#
# 【出力の読み方】
# ・filter()の引数はカンマで区切ると「かつ（AND）」の意味になる
#   （`condition == "voiceless" & VOT > 100` と書いても同じ）。
# ・select()は列を絞るだけでなく、`select(speaker, vot = VOT)`のように
#   列名を変えることもできる。
# ・**元の`vot_data`は変わっていない。** filter()もselect()も、新しい
#   データフレームを返すだけで元のオブジェクトを書き換えない
#   （`vot_data <- vot_data %>% filter(...)`のように代入して初めて上書きされる）。


In [ ]:
# arrange(): 並べ替え。desc()を付けると降順（大きい順）になる。
vot_data %>%
  select(speaker, condition, VOT) %>%
  arrange(desc(VOT)) %>%
  head(5)   # 上位5行だけ表示

# --- 実行結果 ---
# # A tibble: 5 x 3
#   speaker condition   VOT
#   <fct>   <fct>     <dbl>
# 1 S02     voiceless 109.
# 2 S02     voiceless 104.
# 3 S01     voiceless 103.
# 4 S01     voiceless  99.6
# 5 S01     voiceless  99.6
#
# 【出力の読み方】
# ・VOTが最も長い5件はすべてvoiceless条件で、かつS01とS02に集中している。
#   これはS01・S02の話者内ランダム傾き（有声/無声の作り分けの大きさ）が
#   たまたま大きかったことに対応する——第5章のランダム効果の話に直結する。


In [ ]:
# mutate(): 新しい列を作る。case_when()を組み合わせると、
# 複数の条件に応じて値を振り分けられる（Excelのネストしたif文に相当）。
vot_data %>%
  mutate(vot_class = case_when(
    VOT < 30 ~ "短い",
    VOT < 60 ~ "中間",
    TRUE     ~ "長い"          # 上のどれにも当てはまらない場合の既定値
  )) %>%
  count(vot_class)             # count()はgroup_by()+summarise(n=n())の省略形

# --- 実行結果 ---
# # A tibble: 3 x 2
#   vot_class     n
#   <chr>     <int>
# 1 中間         32
# 2 短い         88
# 3 長い         80
#
# 【出力の読み方】
# ・case_when()は上から順に条件を評価し、最初に真になった条件の右辺を採用する。
#   `VOT < 60`の行に「30以上60未満」ではなく「60未満」とだけ書けるのは、
#   その手前で`VOT < 30`がすでに弾かれているためである。
# ・★このままだと3水準の並び順が「中間・短い・長い」とアルファベット順になり、
#   人間の感覚（短い→中間→長い）と一致しない。**順序を意図した並びにしたい
#   ときは、`factor(vot_class, levels = c("短い", "中間", "長い"))`のように
#   明示的にlevelsを指定する。** これを忘れると、後で作る図の凡例や軸の
#   並びも同じ意味のない順序になってしまう。


■ wide形式とlong形式を行き来する（pivot）：

第1章冒頭で「音声データはlong formatで持つ」と述べたが、報告用の表や、他のソフトから受け取ったデータはwide format（話者を行、条件を列に並べた表）のことが多い。`pivot_wider()`と`pivot_longer()`で相互に変換できる。

In [ ]:
# pivot_wider(): long → wide。話者ごとに条件別の平均VOTを求めてから、
# condition列の値（voiced/voiceless）を新しい列名に「展開」する。
wide <- vot_data %>%
  group_by(speaker, condition) %>%
  summarise(mean_VOT = round(mean(VOT), 1), .groups = "drop") %>%
  pivot_wider(names_from = condition, values_from = mean_VOT)

print(wide)

# --- 実行結果 ---
# # A tibble: 10 x 3
#    speaker voiced voiceless
#    <fct>    <dbl>     <dbl>
#  1 S01       27.6      96.1
#  2 S02       16.6      95.6
#  3 S03       22.7      62
#  4 S04       22        78.3
#  5 S05       21.8      71.1
#  6 S06       19.2      75.9
#  7 S07       28.2      83
#  8 S08       20.3      44.1
#  9 S09       30.4      62.7
# 10 S10       19.7      84.9
#
# 【出力の読み方】
# ・`names_from`に指定した列（condition）の各水準が新しい列名になり、
#   `values_from`に指定した列（mean_VOT）の値がそこに入る。1話者1行の、
#   人間が見て比較しやすい表になった。**この形はモデルには渡せない**
#   （`lm(VOT ~ condition, ...)`のような式は、1行1観測のlong formatを前提に
#   しているため）ので、あくまで報告・確認用と割り切る。
# ・話者ごとに voiced と voiceless の差を目で追えるようになった。S08は
#   voiced 20.3・voiceless 44.1と差が小さく、S01は27.6・96.1と差が大きい。
#   この「話者ごとに条件の効果の大きさが違う」性質が、第5章のランダム傾きの
#   実体である。


In [ ]:
# pivot_longer(): wide → long。上で作ったwideを元のlongに戻す。
long_again <- wide %>%
  pivot_longer(cols = c(voiced, voiceless),
               names_to = "condition", values_to = "mean_VOT")

head(long_again, 4)

# --- 実行結果 ---
# # A tibble: 4 x 3
#   speaker condition mean_VOT
#   <fct>   <chr>        <dbl>
# 1 S01     voiced        27.6
# 2 S01     voiceless     96.1
# 3 S02     voiced        16.6
# 4 S02     voiceless     95.6
#
# 【出力の読み方】
# ・`cols`に指定した列（voiced, voiceless）が1つの列（ここではcondition）に
#   まとめられ、その値（mean_VOT）が別の列に落ちる。pivot_wider()の逆操作である。
# ・**wideとlongの間には情報の損失がない**（往復できる）。どちらの形を使うかは
#   「その先で何をしたいか」で決めればよく、迷ったときは統計モデルに渡せる
#   longを基本形にしておくのが安全である。


■ 別の表と結びつける（join）：

実際の研究では、観測データ（vot_data）とは別に、話者ごとの属性をまとめた表（年齢・方言・録音年数など）を持っていることが多い。`left_join()`で2つの表を話者IDをキーにして結びつける。

In [ ]:
# 話者ごとの「発話経験年数」という架空の属性表を作る（本来は別のCSVから読む）。
speaker_info <- tibble(
  speaker   = factor(sprintf("S%02d", 1:10)),
  years_exp = c(2, 15, 8, 1, 20, 5, 12, 3, 9, 25)
)

# left_join(): 左側（vot_data）の全行を残したまま、speakerが一致する行に
# 右側（speaker_info）の列を付け足す。
vot_data %>%
  left_join(speaker_info, by = "speaker") %>%
  distinct(speaker, years_exp)     # 話者ごとに1行だけ表示（確認用）

# --- 実行結果 ---
# # A tibble: 10 x 2
#    speaker years_exp
#    <fct>       <dbl>
#  1 S01             2
#  2 S02            15
#  3 S03             8
#  ...（以下略、10話者分）
#
# 【出力の読み方】
# ・`by = "speaker"`で「どの列をキー（照合の手がかり）にするか」を指定する。
#   両方の表で列名が違う場合は`by = c("speaker" = "id")`のように書く。
# ・この例では`left_join()`後も200行のままである。右側（speaker_info）の
#   `speaker`列に重複が無いため。右側のキーに重複があると、対応する左側の行が
#   その分だけ複製され、結合後の行数は増える。**「行数を変えない」のは右側の
#   キーが一意なときだけ**であり、結合前に`count(speaker_info, speaker) %>%
#   filter(n > 1)`のような形で一意性を確認し、結合後は`nrow()`を検算する
#   習慣をつける。行が増減しては困る場面（測定データに属性を付け足すだけの
#   場面）でこそ、このキー一意性の確認が欠かせない。
# ・結合したあとは、`years_exp`のような話者属性を予測子として
#   `lmer(VOT ~ condition + years_exp + (1 + condition | speaker) + ...)`
#   のようにモデルへ組み込める。第5章以降で扱う混合効果モデルの
#   予測子を増やす、典型的な下ごしらえである。


■ 欠測値（NA）の扱い：

実際の音響測定では、雑音でピッチが取れない、境界が判定できないなどの理由で値が欠ける。欠測を無視して`mean()`を取ると、結果自体が`NA`になる（Rは既定で「欠けたものを含む計算はできない」という安全側の挙動を取る）。

In [ ]:
# 練習用に、元のデータをコピーしてから8箇所をわざと欠測にする。
vot_missing <- vot_data
set.seed(1)
na_idx <- sample(nrow(vot_missing), 8)
vot_missing$VOT[na_idx] <- NA

cat("欠測数:", sum(is.na(vot_missing$VOT)), "\n")

vot_missing %>%
  group_by(condition) %>%
  summarise(n_na = sum(is.na(VOT)), .groups = "drop")

# drop_na(): 指定した列がNAの行をまるごと除く
clean <- vot_missing %>% drop_na(VOT)
cat("drop_na()後の行数:", nrow(clean), "（元は", nrow(vot_missing), "行）\n")

# --- 実行結果 ---
# 欠測数: 8
# # A tibble: 2 x 2
#   condition  n_na
#   <fct>     <int>
# 1 voiced        5
# 2 voiceless     3
# drop_na()後の行数: 192 （元は 200 行）
#
# 【出力の読み方】
# ・`sum(is.na(x))`は「TRUEを1、FALSEを0として合計する」という定石で、
#   欠測の個数を数える最も基本的な書き方である。
# ・`mean(vot_missing$VOT)`をそのまま実行すると結果は`NA`になる
#   （`na.rm = TRUE`を付けて初めて欠測を除いた平均が返る）。うっかり
#   `na.rm = TRUE`を付け続けると、**欠測が増えていることに気づかないまま
#   分析を進めてしまう危険がある**。まず欠測の個数と、それが条件・話者と
#   関係していないかを確認してから対処する習慣が要る（第15章で「情報を
#   持つ欠測」として詳しく扱う）。
# ・`drop_na()`は最も単純な対処（欠測行を丸ごと消す）だが、欠測が条件と
#   偏って結びついている場合はこれだけでは足りない。第15章を参照。


■ 型変換の落とし穴：

CSVを読み込むとき、数値であるはずの列が文字列として読み込まれることがある。よくある原因は、数値の代わりに`"N/A"`や空白のような文字列が1つでも混ざっていることである。readr（`read_csv()`）はその列全体を安全側に倒して文字列型（`<chr>`）として読む。

In [ ]:
# 数値の列に "N/A" が1つ混ざった、よくある「汚れたCSV」を模した例。
messy <- tibble(id = 1:5,
                vot_str = c("28.1", "88.0", "N/A", "30.7", "94.1"))

messy_numeric <- messy %>%
  mutate(vot_num = as.numeric(vot_str))   # 文字列 → 数値に変換

print(messy_numeric)
cat("NAになった行数:", sum(is.na(messy_numeric$vot_num)), "\n")

# --- 実行結果 ---
# Warning message: NAs introduced by coercion
# # A tibble: 5 x 3
#      id vot_str vot_num
#   <int> <chr>     <dbl>
# 1     1 28.1       28.1
# 2     2 88.0       88
# 3     3 N/A        NA
# 4     4 30.7       30.7
# 5     5 94.1       94.1
# NAになった行数: 1
#
# 【出力の読み方】
# ・`as.numeric()`は数値に変換できない文字列（"N/A"など）を黙って`NA`に
#   変える。ただし**"Warning message: NAs introduced by coercion"という
#   警告は必ず出る**。この警告を読み飛ばさないことが重要で、警告が出たら
#   `sum(is.na(...))`で何行がNAになったかを必ず数える。
# ・実務では、`read_csv()`が返す「列の型を推測した結果」の警告
#   （`Warning: 1 parsing failure`のような形で出る）も同じ理由で起きる。
#   **数値であるべき列が`<chr>`のまま読み込まれていたら、まずこの種の
#   混入を疑う。** 第1章冒頭で述べた「表は値より先に型を見る」という
#   習慣は、この落とし穴を早期に発見するためのものである。


■ 可視化（ggplot2）：

平均値だけを見る前に分布を見る習慣をつける（バイオリンプロット＋箱ひげ図）。

In [ ]:
ggplot(vot_data, aes(x = condition, y = VOT, fill = condition)) +
  # geom_violin(): 分布の「形」を左右対称の面で描く（バイオリンプロット）
  geom_violin(alpha = 0.5, trim = FALSE) +
  # geom_boxplot(): 中央値・四分位範囲・外れ値を重ねて描く。widthを小さくしてバイオリンの中に収める
  geom_boxplot(width = 0.2, color = "black", alpha = 0.7) +
  labs(
    title = "Condition別 VOT の分布",
    subtitle = "バイオリンプロットと箱ひげ図の組み合わせ",
    x = "Laryngeal Category (Condition)",
    y = "VOT (ms)"
  ) +
  theme_minimal(base_size = 14, base_family = "notosans") +
  theme(legend.position = "none")  # x軸に凡例と同じ情報が出ているので凡例は消す

# --- 実行結果 ---
# 表ではなく図が1枚描画される（テキスト出力は無い）。
#
# 【図の読み方】
# ・横軸が条件、縦軸がVOT(ms)。左のvoicedの塊は20ms付近、右のvoicelessの塊は
#   75ms付近にあり、2つの塊は**縦軸上でまったく重なっていない**。
#   この「重なっていない」ことが、後の統計で大きなt値が出る理由である。
# ・バイオリン（外側の面）の横幅は、その高さの値がどれくらい多く観測されたかを
#   表す。太いところに値が集中している。voicelessの面はvoicedより縦に長く
#   伸びており、ばらつきが大きいことが形として見える（上の表のsd_VOTに対応）。
# ・内側の細い箱ひげ図は、太い横線が中央値、箱の上下端が第1・第3四分位数。
#   箱の中に観測の半分が入る。ひげの外に描かれる点が外れ値候補である。
# ・trim = FALSE を指定しているので、バイオリンの裾は実際の最小値・最大値より
#   外まで滑らかに伸びる。**裾の先端を「最大値」と読んではいけない**
#   （既定の trim = TRUE ならデータの範囲で切り落とされる）。
# ・図を出す目的は見栄えではなく、平均を計算する前に「2群が本当に分かれて
#   いるか」「片方だけ極端に歪んでいないか」を目で確かめることにある。


■ 演習：

1. `group_by(speaker)` に変えて話者ごとの平均VOTを集計せよ。
2. `geom_jitter()` を重ねて個々の観測点を表示せよ。
3. `F0` についても同様の要約統計・可視化を行え。

■ 【発展】自分のデータを使う場合：Google Driveのマウント：

本書の第0〜9章は、必要なものがGoogleアカウントだけで済むように、すべてのデータをその場で作る（実測データをどこにも置かない）設計にしてある。だが自分の研究では、当然ながら手元の録音から測定した本物のCSVを読み込みたくなる。そのときだけ必要になるのが**Google Driveのマウント**である。

**手順**
1. Colab画面の左端にあるフォルダのアイコン（ファイル）をクリックする
2. 一番上のフォルダアイコンに斜線が入ったようなボタン（「ドライブをマウント」）をクリックする——これはRのコードではなく、Colabの画面のボタン操作でよい
3. Googleアカウントでのアクセス許可を求められるので許可する
4. 少し待つと、ファイル一覧に`drive`というフォルダが現れる。これで自分のGoogle Drive全体が、この実行環境から`/content/drive/MyDrive/`というパスで見えるようになる

**読み込み方**（ボタン操作のあとはコードは通常のread_csv()と同じ）
```r
my_data <- read_csv("/content/drive/MyDrive/フォルダ名/measurements.csv")
```

**注意点**
- マウントは**そのセッション限り**。ランタイムが切断・再起動されると毎回マウントし直しになる（許可自体は次回以降スキップされることが多い）
- マウントすると、このノートブックのコード次第で**Drive内のファイルを書き換えたり削除したりできてしまう**。他人と共有するノートブックには、自分のDriveを書き換える処理を安易に入れないこと
- ノートブックを他の人（例えば共著者）と共有する場合、コード中に書いた`/content/drive/MyDrive/...`のパスは**自分のDrive構成に依存する**。共有相手の環境では同じパスにファイルがない可能性が高いので、パスを変数化してセルの先頭にまとめておくと親切
- 音声ファイル（.wav等）を大量に読み込む処理は、Driveとのやり取りがネットワーク経由になるため、ローカルディスクより遅いことがある。大きなコーパスを繰り返し処理する場合は、`speech-corpus-engineering`系のワークフロー（バッチ処理・チェックポイント運用）を検討するとよい

■ 演習の解答：

3問とも、本文で使った `group_by()` → `summarise()` と ggplot2 の積み上げ方をそのまま応用したものである。

In [ ]:
# 1. 話者ごとの平均VOTを集計する。
#    group_by()に渡す列を変えるだけで、集計の単位が変わる。
vot_data %>%
  group_by(speaker) %>%
  summarise(mean_VOT = mean(VOT), sd_VOT = sd(VOT), n = n(), .groups = "drop")

# 2. 箱ひげ図に個々の観測点を重ねる。
#    geom_jitter()は点を左右に少しずらして重なりを避ける（jitter=ゆらぎ）。
#    width=0.15くらいにしないと、隣の条件の位置まで点が流れて誤解を招く。
ggplot(vot_data, aes(x = condition, y = VOT, fill = condition)) +
  geom_boxplot(outlier.shape = NA) +   # 外れ値の点は消す（下のjitterと二重表示になるため）
  geom_jitter(width = 0.15, alpha = 0.4, size = 1) +
  theme_minimal(base_size = 14, base_family = "notosans")

# 3. F0について同じことをする。
#    VOTと違い、F0は条件と無関係に生成してあるので「差が無い」結果になるはず。
#    差が出ない変数を1つ持っておくと、手法が差を捏造していないか確認できる。
vot_data %>%
  group_by(condition) %>%
  summarise(mean_F0 = mean(F0), sd_F0 = sd(F0), .groups = "drop")

ggplot(vot_data, aes(x = condition, y = F0, fill = condition)) +
  geom_boxplot(outlier.shape = NA) +
  geom_jitter(width = 0.15, alpha = 0.4, size = 1) +
  theme_minimal(base_size = 14, base_family = "notosans")

# --- 実行結果（1問目と3問目の表。2問目と3問目の図は画面に描画される）---
# # A tibble: 10 x 4
#    speaker mean_VOT sd_VOT     n
#    <chr>      <dbl>  <dbl> <int>
#  1 S01         61.8   35.6    20
#  2 S02         56.1   41.0    20
#  3 S03         42.4   21.2    20
#  4 S04         50.2   29.8    20
#  5 S05         46.5   26.4    20
#  6 S06         47.6   29.9    20
#  7 S07         55.6   28.8    20
#  8 S08         32.2   13.4    20
#  9 S09         46.6   18.8    20
# 10 S10         52.3   33.8    20
#
# # A tibble: 2 x 3
#   condition mean_F0 sd_F0
#   <chr>       <dbl> <dbl>
# 1 voiced       130.  15.0
# 2 voiceless    129.  14.0
#
# 【出力の読み方】
# ・1問目の表で最も重要なのは **話者ごとの平均VOTが32.2msから61.8msまで
#   30ms近くばらついている**ことである。全員が同じ20語を同じ条件で読んでいるのに
#   これだけ違う。「話者を無視して200行を独立に扱ってはいけない」という
#   第5章の出発点が、この1枚の表にそのまま出ている。
# ・n列がどの話者も20で揃っているのは、デザインがバランスしている（各話者が
#   全単語を1回ずつ）ことの確認である。ここが揃っていなければ、集計の前に
#   データの取りこぼしを疑う。
# ・sd_VOTも13.4〜41.0と話者ごとに違う。これは有声/無声の作り分けの大きさが
#   話者によって違うため（第5章のランダム傾きに対応する）。
# ・3問目のF0の表は、2条件の平均が130.と129.でほぼ同じ、SDもほぼ同じ。
#   **差が無いものを分析すれば、ちゃんと「差が無い」と出る**ことの確認である。
#   VOTで見えた差が手法の副産物ではないという保証になる。
# ・数値の末尾が「130.」のように点で終わっているのは桁の省略記号ではなく、
#   tibbleが有効数字3桁で丸めて表示しているためである（実際の値は130.0…）。
# ・2問目・3問目の図では、箱ひげ図の上に個々の点が重なって表示される。
#   VOTの図は2条件の点の塊がはっきり離れ、F0の図は完全に重なる。
